# 02 Silver validation

Curate, deduplicate and validate the bronze rows, then inspect the
deduplication effect and the electoral-period distribution.

Run this notebook from PyCharm using the IDE **Managed Jupyter server**:
`Project Settings > Python Interpreter > Jupyter`, let PyCharm start and
manage the server, and select the project `.venv` as the kernel. No separate
`jupyter notebook` process is required (see `notebooks/README.md`).

## Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Make the project importable no matter where the kernel starts.
_here = Path.cwd()
_root = _here.parent if _here.name == "notebooks" else _here
sys.path.insert(0, str(_root))
os.chdir(_root)

# Force SQLite so notebooks never touch PostgreSQL by accident.
os.environ["ETL_DB_BACKEND"] = "sqlite"
os.environ.setdefault("ETL_SQLITE_PATH", "data/warehouse/violence_staging.db")

import pandas as pd  # noqa: F401  (used by later cells)

from violence_etl.settings import get_settings

settings = get_settings()
print("backend:", settings.backend)
print("sqlite :", settings.sqlite_path)
print("bronze :", settings.bronze_dir)

## Build silver from bronze

Uses the landing file already produced by notebook 01 (or the CLI).

In [ ]:
import glob
import os

from violence_etl.bronze import ingest_bronze
from violence_etl.db import make_engine
from violence_etl.models import Base
from violence_etl.silver import build_silver

engine = make_engine(settings=settings)
Base.metadata.create_all(engine)

landing = sorted(glob.glob(os.path.join(settings.bronze_dir, "landing", "*.csv")))
assert landing, "No landing file found. Run notebook 01 or `cli download` first."
landing_path = landing[-1]
print("using:", landing_path)

ingest_bronze(engine, settings, csv_path=landing_path)
result = build_silver(engine, settings)
result

## Deduplication effect

Default `ETL_DEDUP_MODE=drop`: 15,696 raw rows collapse to 14,212 silver
rows (1,484 exact duplicates removed), total crimes 15,696 -> 14,212.

In [ ]:
print("raw_rows            :", result["raw_rows"])
print("silver_rows         :", result["silver_rows"])
print("dropped_duplicates  :", result["dropped_duplicates"])
print("total_crimes        :", result["total_crimes"])
print("dedup_mode          :", result["dedup_mode"])

## Rejected rows

Nothing is silently dropped; every reject carries a reason.

In [ ]:
pd.read_sql(
    "select reject_reason, count(*) as rows "
    "from silver_rejected_rows group by reject_reason order by rows desc",
    engine,
)

## Rows and crimes by electoral period

In [ ]:
pd.read_sql(
    "select electoral_period, count(*) as rows, sum(cantidad) as crimes "
    "from silver_crimes group by electoral_period order by electoral_period",
    engine,
)

## Rows and crimes per year (with the period label)

Zero unmapped years in 2003..2026.

In [ ]:
pd.read_sql(
    "select year, electoral_period, count(*) as rows, sum(cantidad) as crimes "
    "from silver_crimes group by year, electoral_period order by year",
    engine,
)

In [ ]:
# Guard rails: codes are never numeric and every year is classified.
pd.read_sql(
    "select cod_depto, cod_muni, typeof(cod_depto) as depto_type "
    "from silver_crimes limit 3",
    engine,
)